# Results for model: mistralai/mistral-nemotron

In [1]:
import polars as pl
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering: Calculate global TOP_QUANTILE of 'feature_00' relative to 'responder_6'
TOP_QUANTILE = 0.15
rolling_batches = 5

# Process data in rolling batches
for i in range(0, df.select('date_id').unique().n_unique(), rolling_batches):
    batch_dates = df.filter(pl.col('date_id').is_in(
        df.select('date_id').unique().to_list()[i:i+rolling_batches]
    ))

    if batch_dates.n_rows == 0:
        continue

    # Calculate TOP_QUANTILE
    quantile_value = batch_dates.select(
        pl.col('feature_00').quantile(TOP_QUANTILE)
    ).item()

    df = df.with_columns(
        pl.col('top_quantile_feature00').fill_null(
            pl.lit(quantile_value * batch_dates.select(
                pl.col('responder_6').mean().item()
            ))
        )
    )

# Prepare features and target
features = [
    col for col in df.columns
    if col.startswith('feature_') or col == 'top_quantile_feature00'
]

X = df.select(features).to_numpy()
y = df.select('responder_6').to_numpy().flatten()

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost model
xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    learning_rate=0.1,
    max_depth=6,
    n_estimators=100,
    random_state=42,
    verbose=0
)

xgb_reg.fit(X_train, y_train)

# Evaluate
y_pred = xgb_reg.predict(X_val)
msle = mean_squared_log_error(y_val + 1, y_pred + 1)
print(f'MSLE on validation set: {np.sqrt(msle)}')

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet